In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))
print(ROOT)

from database.connectie.connectie import get_engine,getData

C:\Users\Luizv\Documents\HoGent\jaar3\BachelorProef\BachelorProef2026


In [3]:
engine = get_engine()

In [ ]:
query_market = """
    SELECT
        fm.DateKey,
        dd.FullDateAlternateKey   AS TradeDate,
        fm.StockKey,
        fm.[Open],
        fm.High,
        fm.Low,
        fm.[Close],
        fm.Volume
    FROM FactMarketData fm
    JOIN DimDate dd ON fm.DateKey = dd.DateKey
    ORDER BY fm.StockKey, dd.FullDateAlternateKey
"""

In [ ]:
df_market = getData(engine, query_market)
df_market['TradeDate'] = pd.to_datetime(df_market['TradeDate'])


In [ ]:
query_econ = """
    SELECT
        fe.DateKey,
        dd.FullDateAlternateKey AS TradeDate,
        fe.USD, fe.OIL, fe.VIX, fe.YieldSpread,
        fe.InfExpectation, fe.FinStress,
        fe.FedFundsRate, fe.FedBalanceSheet,
        fe.CPI, fe.PPI, fe.Consumer_Confidence
    FROM FactEcon fe
    JOIN DimDate dd ON fe.DateKey = dd.DateKey
    ORDER BY dd.FullDateAlternateKey
"""

In [ ]:
df_econ = getData(engine, query_econ)
df_econ['TradeDate'] = pd.to_datetime(df_econ['TradeDate'])

In [ ]:
query_news = """
    SELECT
        fn.DateKey,
        dd.FullDateAlternateKey AS TradeDate,
        AVG(fn.PositivityScore) AS AvgNewsSentiment,
        AVG(fn.influenceScore)  AS AvgNewsInfluence,
        COUNT(*)                AS NewsCount
    FROM FactNews fn
    JOIN DimDate dd ON fn.DateKey = dd.DateKey
    GROUP BY fn.DateKey, dd.FullDateAlternateKey
    ORDER BY dd.FullDateAlternateKey
"""

In [ ]:
df_news = getData(engine, query_news)
df_news['TradeDate'] = pd.to_datetime(df_news['TradeDate'])


In [ ]:
query_twitter = """
    SELECT
        dt.DateKey,
        dd.FullDateAlternateKey AS TradeDate,
        AVG(dt.PositivityScore)  AS AvgTweetSentiment,
        AVG(dt.influenceScore)   AS AvgTweetInfluence,
        COUNT(*)                 AS TweetCount
    FROM DimTwitter dt
    JOIN DimDate dd ON dt.DateKey = dd.DateKey
    GROUP BY dt.DateKey, dd.FullDateAlternateKey
    ORDER BY dd.FullDateAlternateKey
"""

In [ ]:
df_twitter = getData(engine, query_twitter)
df_twitter['TradeDate'] = pd.to_datetime(df_twitter['TradeDate'])

In [ ]:
from sklearn.preprocessing import RobustScaler
import joblib

In [ ]:
def build_feature_matrix(df_market, df_econ, df_news, df_twitter):
    df = df_market.copy()

    # Merge econ
    df = df.merge(
        df_econ.drop(columns=['DateKey']),
        on='TradeDate', how='left'
    )
    # Merge news
    df = df.merge(
        df_news.drop(columns=['DateKey']),
        on='TradeDate', how='left'
    )
    # Merge twitter
    df = df.merge(
        df_twitter.drop(columns=['DateKey']),
        on='TradeDate', how='left'
    )

    df = df.sort_values(['StockKey', 'TradeDate']).reset_index(drop=True)
    return df

In [ ]:
def add_technical_indicators(df):
    grp = df.groupby('StockKey')

    # Returns
    df['Return_1d']  = grp['Close'].pct_change(1)
    df['Return_5d']  = grp['Close'].pct_change(5)
    df['Return_10d'] = grp['Close'].pct_change(10)

    # Moving averages
    df['MA_5']  = grp['Close'].transform(lambda x: x.rolling(5).mean())
    df['MA_20'] = grp['Close'].transform(lambda x: x.rolling(20).mean())
    df['MA_50'] = grp['Close'].transform(lambda x: x.rolling(50).mean())

    # MA crossover signaal
    df['MA_cross_5_20']  = (df['MA_5']  - df['MA_20'])  / df['MA_20']
    df['MA_cross_20_50'] = (df['MA_20'] - df['MA_50'])  / df['MA_50']

    # Volatiliteit (rolling std van returns)
    df['Vol_5d']  = grp['Return_1d'].transform(lambda x: x.rolling(5).std())
    df['Vol_20d'] = grp['Return_1d'].transform(lambda x: x.rolling(20).std())

    # RSI (14 periodes)
    def calc_rsi(series, window=14):
        delta = series.diff()
        gain  = delta.clip(lower=0).rolling(window).mean()
        loss  = (-delta.clip(upper=0)).rolling(window).mean()
        rs    = gain / (loss + 1e-9)
        return 100 - (100 / (1 + rs))

    df['RSI_14'] = grp['Close'].transform(calc_rsi)

    # Bollinger Bands positie
    bb_mid = grp['Close'].transform(lambda x: x.rolling(20).mean())
    bb_std = grp['Close'].transform(lambda x: x.rolling(20).std())
    df['BB_position'] = (df['Close'] - bb_mid) / (2 * bb_std + 1e-9)

    # Volume ratio (huidig vs 20d gemiddelde)
    df['Volume_ratio'] = df['Volume'] / (
        grp['Volume'].transform(lambda x: x.rolling(20).mean()) + 1e-9
    )

    # Price range (High-Low) als % van Close
    df['DayRange_pct'] = (df['High'] - df['Low']) / (df['Close'] + 1e-9)

    return df

In [ ]:
# ─────────────────────────────────────────
# 2C. Label: spike/dip definitie
#     1 = spike (grote stijging morgen)
#    -1 = dip   (grote daling morgen)
#     0 = neutraal
#
#     Drempel: |return| > 1.5x rolling std
#     → past zich aan per stock per periode
# ─────────────────────────────────────────
def add_labels(df, threshold_multiplier=1.5, horizon=1):
    """
    horizon=1 → voorspel return van morgen
    threshold_multiplier → hoe 'groot' moet de beweging zijn
    """
    grp = df.groupby('StockKey')

    # Toekomstige return
    df['FutureReturn'] = grp['Close'].transform(
        lambda x: x.pct_change(horizon).shift(-horizon)
    )

    # Dynamische drempel per stock (rolling 60d std van returns)
    df['ReturnStd_60d'] = grp['Return_1d'].transform(
        lambda x: x.rolling(60).std()
    )
    df['Threshold'] = threshold_multiplier * df['ReturnStd_60d']

    # Label
    conditions = [
        df['FutureReturn'] >  df['Threshold'],   # spike
        df['FutureReturn'] < -df['Threshold'],   # dip
    ]
    df['Label'] = np.select(conditions, [1, -1], default=0)

    # Binair alternatief (0/1): spike of dip vs neutraal
    df['Label_binary'] = (df['Label'] != 0).astype(int)

    return df


In [ ]:
# ─────────────────────────────────────────
# 2D. Alles samenvoegen + NaN droppen
# ─────────────────────────────────────────
FEATURE_COLS = [
    # Prijs-gebaseerd
    'Return_1d', 'Return_5d', 'Return_10d',
    'MA_cross_5_20', 'MA_cross_20_50',
    'Vol_5d', 'Vol_20d',
    'RSI_14', 'BB_position', 'Volume_ratio', 'DayRange_pct',
    # Macro
    'USD', 'OIL', 'VIX', 'YieldSpread',
    'InfExpectation', 'FinStress',
    'FedFundsRate', 'CPI', 'PPI', 'Consumer_Confidence',
    # Sentiment
    'AvgNewsSentiment', 'AvgNewsInfluence', 'NewsCount',
    'AvgTweetSentiment', 'AvgTweetInfluence', 'TweetCount',
]

In [ ]:
def prepare_dataset(df):
    df = df.dropna(subset=FEATURE_COLS + ['Label'])
    return df

In [ ]:
# ─────────────────────────────────────────
# 2E. Scaler fitten & opslaan
#     (BELANGRIJK: scaler wordt gefit op train set only)
# ─────────────────────────────────────────
def fit_and_save_scaler(df_train, feature_cols, path='scaler.joblib'):
    scaler = RobustScaler()
    scaler.fit(df_train[feature_cols])
    joblib.dump(scaler, path)
    print(f"✅ Scaler opgeslagen: {path}")
    return scaler

In [ ]:
# step3_model.py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# ─────────────────────────────────────────
# 3A. Sequence dataset
#     Elke sample = venster van SEQ_LEN dagen
# ─────────────────────────────────────────
class StockSequenceDataset(Dataset):
    def __init__(self, df, feature_cols, label_col, seq_len=30):
        """
        df        : gesorteerd op (StockKey, TradeDate)
        seq_len   : aantal terugkijkdagen per sample
        label_col : 'Label' (3 klassen) of 'Label_binary' (2 klassen)
        """
        self.seq_len = seq_len
        self.samples = []

        for stock, grp in df.groupby('StockKey'):
            grp = grp.sort_values('TradeDate').reset_index(drop=True)
            X = grp[feature_cols].values.astype(np.float32)
            y = grp[label_col].values.astype(np.int64)

            # Verschuif label zodat -1,0,1 → 0,1,2 voor CrossEntropy
            if label_col == 'Label':
                y = y + 1  # -1→0 (dip), 0→1 (neutraal), 1→2 (spike)

            for i in range(seq_len, len(X)):
                self.samples.append((
                    X[i - seq_len : i],   # shape: (seq_len, n_features)
                    y[i]
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        X, y = self.samples[idx]
        return torch.tensor(X), torch.tensor(y)


In [ ]:
# ─────────────────────────────────────────
# 3B. LSTM model
# ─────────────────────────────────────────
class SpikeDipLSTM(nn.Module):
    def __init__(
        self,
        n_features,
        hidden_size=128,
        num_layers=2,
        n_classes=3,       # 3 = dip/neutraal/spike  |  2 = binary
        dropout=0.3
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)    # → logits
        )

    def forward(self, x):
        # x shape: (batch, seq_len, n_features)
        lstm_out, _ = self.lstm(x)
        last_hidden  = lstm_out[:, -1, :]   # laatste tijdstap
        logits       = self.classifier(last_hidden)
        return logits

In [ ]:
# step4_train.py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd
import joblib

# from step2_build_dataset import (
#     build_feature_matrix, add_technical_indicators,
#     add_labels, prepare_dataset, fit_and_save_scaler,
#     FEATURE_COLS
# )
# from step3_model import StockSequenceDataset, SpikeDipLSTM

In [ ]:
# ─── Hyperparameters ──────────────────────
SEQ_LEN      = 30
BATCH_SIZE   = 64
EPOCHS       = 50
LR           = 1e-3
HIDDEN       = 128
LAYERS       = 2
DROPOUT      = 0.3
N_CLASSES    = 3        # 3 = dip/neutraal/spike
LABEL_COL    = 'Label'  # of 'Label_binary' voor binair
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Training op: {DEVICE}")

In [ ]:
# ─── Data laden (vanuit stap 1 resultaten) ─
# (sla de DataFrames op als parquet na stap 1)
df_market  = pd.read_parquet('df_market.parquet')
df_econ    = pd.read_parquet('df_econ.parquet')
df_news    = pd.read_parquet('df_news.parquet')
df_twitter = pd.read_parquet('df_twitter.parquet')


In [ ]:
df = build_feature_matrix(df_market, df_econ, df_news, df_twitter)
df = add_technical_indicators(df)
df = add_labels(df)
df = prepare_dataset(df)

In [ ]:
# ─── Train / Val / Test split (tijdsgebaseerd!) ─
# Nooit random split bij tijdreeksen
df = df.sort_values(['StockKey', 'TradeDate'])
dates = df['TradeDate'].unique()
n     = len(dates)

In [ ]:
train_cutoff = dates[int(n * 0.70)]
val_cutoff   = dates[int(n * 0.85)]

In [ ]:
df_train = df[df['TradeDate'] <= train_cutoff]
df_val   = df[(df['TradeDate'] > train_cutoff) & (df['TradeDate'] <= val_cutoff)]
df_test  = df[df['TradeDate'] > val_cutoff]


In [ ]:
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")


In [ ]:
scaler = fit_and_save_scaler(df_train, FEATURE_COLS)


In [ ]:
df_train[FEATURE_COLS] = scaler.transform(df_train[FEATURE_COLS])
df_val[FEATURE_COLS]   = scaler.transform(df_val[FEATURE_COLS])
df_test[FEATURE_COLS]  = scaler.transform(df_test[FEATURE_COLS])


In [ ]:
train_ds = StockSequenceDataset(df_train, FEATURE_COLS, LABEL_COL, SEQ_LEN)
val_ds   = StockSequenceDataset(df_val,   FEATURE_COLS, LABEL_COL, SEQ_LEN)
test_ds  = StockSequenceDataset(df_test,  FEATURE_COLS, LABEL_COL, SEQ_LEN)


In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)


In [ ]:
all_labels = [s[1].item() for s in train_ds]
classes    = np.unique(all_labels)
weights    = compute_class_weight('balanced', classes=classes, y=all_labels)
weights_t  = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
print(f"Class weights: {dict(zip(classes, weights.round(3)))}")


In [ ]:
model     = SpikeDipLSTM(len(FEATURE_COLS), HIDDEN, LAYERS, N_CLASSES, DROPOUT).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights_t)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

In [ ]:
best_val_loss = float('inf')


In [ ]:
for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item() * len(y_batch)
        preds       = logits.argmax(dim=1)
        correct    += (preds == y_batch).sum().item()
        total      += len(y_batch)

    train_loss /= total
    train_acc   = correct / total

    # --- Validatie ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits    = model(X_batch)
            loss      = criterion(logits, y_batch)
            val_loss += loss.item() * len(y_batch)
            preds     = logits.argmax(dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total   += len(y_batch)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    scheduler.step(val_loss)

    # Beste model opslaan
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  💾 Nieuw beste model opgeslagen (val_loss={val_loss:.4f})")

    if epoch % 5 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:3d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
            f"Val Loss:   {val_loss:.4f} Acc: {val_acc:.3f}"
        )

In [ ]:
print(f"\n✅ Training klaar. Beste val_loss: {best_val_loss:.4f}")


In [ ]:
# step5_evaluate.py
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns # pip install seaborn
import matplotlib.pyplot as plt

# from step3_model import SpikeDipLSTM
# from step4_train import test_loader, FEATURE_COLS, N_CLASSES, HIDDEN, LAYERS, DROPOUT, DEVICE


In [ ]:
model = SpikeDipLSTM(len(FEATURE_COLS), HIDDEN, LAYERS, N_CLASSES, DROPOUT).to(DEVICE)
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
model.eval()

In [ ]:
all_preds, all_labels, all_probs = [], [], []


In [ ]:
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        logits  = model(X_batch)
        probs   = torch.softmax(logits, dim=1)
        preds   = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())
        all_probs.extend(probs.cpu().numpy())


In [ ]:
target_names = ['Dip', 'Neutraal', 'Spike']


In [ ]:
print(classification_report(all_labels, all_preds, target_names=target_names))


In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - Test Set')
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()